# DOWNLOAD GRADIO

In [ ]:
# INSTALL REQUIRED LIBRARIES

!pip install gradio matplotlib pandas -q


REQUIRED LIBRARIES

In [ ]:
# IMPORT LIBRARIES

import gradio as gr
import random
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

MAIN PROGRAM


In [ ]:
# PAYOFF MATRIX

# (Player A payoff, Player B payoff)

payoff_matrix = {
    ("Cooperate", "Cooperate"): (3, 3),
    ("Cooperate", "Betray"): (0, 5),
    ("Betray", "Cooperate"): (5, 0),
    ("Betray", "Betray"): (1, 1),
}


# AI STRATEGIES


def always_cooperate(history_self, history_opponent):
    return "Cooperate"

def always_betray(history_self, history_opponent):
    return "Betray"

def random_strategy(history_self, history_opponent):
    return random.choice(["Cooperate", "Betray"])

def mostly_cooperate(history_self, history_opponent):
    return "Cooperate" if random.random() < 0.8 else "Betray"

def mostly_betray(history_self, history_opponent):
    return "Betray" if random.random() < 0.8 else "Cooperate"

def tit_for_tat(history_self, history_opponent):
    if not history_opponent:
        return "Cooperate"
    return history_opponent[-1]

def grudger(history_self, history_opponent):

    if "Betray" in history_opponent:
        return "Betray"

    return "Cooperate"

def pavlov(history_self, history_opponent):

    if not history_self:
        return "Cooperate"

    if history_self[-1] == history_opponent[-1]:
        return history_self[-1]

    return "Betray" if history_self[-1] == "Cooperate" else "Cooperate"


# STRATEGY DICTIONARY


strategies = {
    "Always Cooperate": always_cooperate,
    "Always Betray": always_betray,
    "Random": random_strategy,
    "Mostly Cooperate": mostly_cooperate,
    "Mostly Betray": mostly_betray,
    "Tit For Tat": tit_for_tat,
    "Grudger": grudger,
    "Pavlov": pavlov,
}


# MAIN SIMULATION FUNCTION

def simulate(
    strategy_a_name,
    strategy_b_name,
    rounds,
    human_move_a,
    human_move_b
):

    history_a = []
    history_b = []

    score_a = 0
    score_b = 0

    cooperation_count = 0
    betrayal_count = 0

    round_scores_a = []
    round_scores_b = []

    result_text = ""

    # Store match history for CSV export
    records = []

    for round_num in range(1, rounds + 1):


        # PLAYER A MOVE


        if strategy_a_name == "Human":
            move_a = human_move_a
        else:
            move_a = strategies[strategy_a_name](
                history_a,
                history_b
            )


        # PLAYER B MOVE


        if strategy_b_name == "Human":
            move_b = human_move_b
        else:
            move_b = strategies[strategy_b_name](
                history_b,
                history_a
            )

        # Save history
        history_a.append(move_a)
        history_b.append(move_b)

        # Get payoff
        payoff_a, payoff_b = payoff_matrix[(move_a, move_b)]

        score_a += payoff_a
        score_b += payoff_b

        round_scores_a.append(score_a)
        round_scores_b.append(score_b)

        # Cooperation stats
        if move_a == "Cooperate" and move_b == "Cooperate":
            cooperation_count += 1

        if move_a == "Betray" or move_b == "Betray":
            betrayal_count += 1

        # Round result
        result_text += (
            f"========== ROUND {round_num} ==========\n"
            f"Player A Move : {move_a}\n"
            f"Player B Move : {move_b}\n"
            f"Round Score   : A={payoff_a} | B={payoff_b}\n"
            f"Total Score   : A={score_a} | B={score_b}\n\n"
        )

        # Save record
        records.append({
            "Round": round_num,
            "Player A Move": move_a,
            "Player B Move": move_b,
            "Player A Score": payoff_a,
            "Player B Score": payoff_b,
            "Total A": score_a,
            "Total B": score_b,
        })


    # WINNER


    if score_a > score_b:
        winner = "🏆 PLAYER A WINS"
    elif score_b > score_a:
        winner = "🏆 PLAYER B WINS"
    else:
        winner = "🤝 MATCH DRAW"


    # STATISTICS


    cooperation_rate = (cooperation_count / rounds) * 100
    betrayal_rate = (betrayal_count / rounds) * 100

    result_text += (
        f"\n==============================\n"
        f"FINAL RESULT\n"
        f"==============================\n\n"
        f"Player A Final Score : {score_a}\n"
        f"Player B Final Score : {score_b}\n\n"
        f"{winner}\n\n"
        f"Cooperation Rate : {cooperation_rate:.2f}%\n"
        f"Betrayal Rate    : {betrayal_rate:.2f}%\n"
    )


    # GRAPH


    plt.figure(figsize=(10,6))

    plt.plot(
        range(1, rounds + 1),
        round_scores_a,
        marker='o',
        linewidth=2
    )

    plt.plot(
        range(1, rounds + 1),
        round_scores_b,
        marker='o',
        linewidth=2
    )

    plt.xlabel("Rounds")
    plt.ylabel("Cumulative Score")
    plt.title("Prisoner's Dilemma Score Progression")

    plt.legend([
        "Player A",
        "Player B"
    ])

    plt.grid(True)

    plt.tight_layout()

    graph_path = "score_graph.png"

    plt.savefig(graph_path)

    plt.close()


    # EXPORT CSV


    df = pd.DataFrame(records)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    csv_path = f"match_results_{timestamp}.csv"

    df.to_csv(csv_path, index=False)


    # RETURN OUTPUTS


    return (
        result_text,
        graph_path,
        csv_path
    )


# CUSTOM CSS (DARK MODE STYLE)


custom_css = """
body {
    background-color: #0f172a;
}

.gradio-container {
    background: linear-gradient(135deg, #0f172a, #1e293b);
    color: white !important;
}

/* ALL TEXT WHITE */
h1, h2, h3, h4, h5, h6,
p, span, label, li, div {
    color: white !important;
}

/* Textboxes */
textarea, input {
    color: black !important;
}

/* Dropdown text */
select {
    color: black !important;
}

/* Buttons */
button {
    font-weight: bold !important;
}
"""



# GRADIO UI


with gr.Blocks(css=custom_css) as app:


    # TITLE


    gr.Markdown(
        """
        # 🎮 Prisoner's Dilemma AI Simulator

        ## Advanced Game Theory Simulation

        ### Features:
        - AI vs AI
        - AI vs Human
        - Multiple Strategies
        - Score Graph
        - CSV Export
        - Winner Detection
        - Cooperation Statistics
        - Dark Mode UI
        """
    )


    # PLAYER SELECTION


    with gr.Row():

        strategy_a = gr.Dropdown(
            choices=list(strategies.keys()) + ["Human"],
            value="Tit For Tat",
            label="Player A Strategy"
        )

        strategy_b = gr.Dropdown(
            choices=list(strategies.keys()) + ["Human"],
            value="Random",
            label="Player B Strategy"
        )


    # HUMAN CONTROLS


    with gr.Row():

        human_move_a = gr.Radio(
            choices=["Cooperate", "Betray"],
            value="Cooperate",
            label="Human Move (Player A)"
        )

        human_move_b = gr.Radio(
            choices=["Cooperate", "Betray"],
            value="Cooperate",
            label="Human Move (Player B)"
        )


    # ROUND SLIDER


    rounds = gr.Slider(
        minimum=1,
        maximum=100,
        value=10,
        step=1,
        label="Number of Rounds"
    )


    # BUTTONS


    with gr.Row():

        run_button = gr.Button("▶ Run Simulation")

        clear_button = gr.Button("🗑 Clear")


    # OUTPUTS


    output_text = gr.Textbox(
        label="Simulation Output",
        lines=25
    )

    output_graph = gr.Image(
        label="Score Progression Graph"
    )

    csv_output = gr.File(
        label="Download Match Results CSV"
    )

    # BUTTON FUNCTIONS


    run_button.click(
        simulate,
        inputs=[
            strategy_a,
            strategy_b,
            rounds,
            human_move_a,
            human_move_b
        ],
        outputs=[
            output_text,
            output_graph,
            csv_output
        ]
    )

    clear_button.click(
        lambda: ("", None, None),
        outputs=[
            output_text,
            output_graph,
            csv_output
        ]
    )


# LAUNCH APP


app.launch(
    debug=True,
    share=True
)

/tmp/ipykernel_18391/4267199810.py:293: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abd22ca58e904ce2a7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
